🌐 [English](dx_app.ipynb) | [한국어](dx_app.ko.ipynb) | **日本語** | [中文](dx_app.zh.ipynb)

# DEEPX Tutorial 02 - DX-APP の使い方

この 2 つ目のチュートリアルでは、DX-APP を紹介し、コンパイル済みの DXNN モデルを AI アプリケーションで実行する方法を学びます。

## 学習目標

このチュートリアルを終えると、次のことができるようになります。

- DX-APP が **[DEEPX Model Zoo](https://developer.deepx.ai/modelzoo/) のすべてのモデル**(497 個のモデルバリアント)に対してすぐに実行できる C++ と Python のサンプルコードを提供していることを理解し、そのコードを使って任意のモデルを実行する
- サンプルがタスク、モデルファミリー、バリアントによってどのように整理されているか、そしてバリアント名がモデルファイル、実行ファイル、スクリプトをどのように結び付けているかを説明する
- チュートリアルに必要なモデルと動画だけをダウンロードする
- 分類、検出、姿勢推定、セグメンテーションのサンプルを実行し、保存された結果をノートブック内で確認する
- 同期、非同期、PPU の各実行方式を使い分け、どの Python バリアントが自分のモデルに適しているかを計測する
- `dx_tool.sh` でサンプルを検索し、デモランチャーを起動する
- スタンドアロンの C++ または Python サンプルをエクスポートし、そのモデルカードを読み、DX-APP ツリーの外で実行する
- コードではなく JSON ファイルを編集することで、マルチモデルグラフを使って複数のモデルを 1 つのパイプラインに連結する

## DX-APP とは?

**DX-APP** は DEEPX NPU 向けのリファレンスアプリケーションです。その最大の特徴は網羅性です。**[DEEPX Model Zoo](https://developer.deepx.ai/modelzoo/) に公開されているすべてのモデルに、対応するすぐに実行可能なサンプル**が C++ と Python の両方で用意されており、タスク別(object detection、classification、face detection、pose、segmentation、depth など)に整理されています。Model Zoo のページでモデルを選べば、それを実行するコードはすでに DX-APP に含まれています。DX-APP は開発者がランタイム環境を素早くセットアップするのを助け、
独自の AI アプリケーションを構築・カスタマイズするためのテンプレートとして機能します。

このチュートリアルでは、DX-All Suite v2.5.0 に同梱されているバージョンである **DX-APP v3.3.0** を使用します。

詳細は [dx_app git repo](https://github.com/DEEPX-AI/dx_app/blob/main/README.md) の概要を参照するか、[こちら](https://developer.deepx.ai/download/?id=580)から DX-APP ユーザーガイドをダウンロードしてください!

> **注:** ユーザーガイドをダウンロードするには、まず https://developer.deepx.ai/ にログインする必要があります。

DX-APP のファイル構成を見てみましょう。

In [ ]:
# Load the shared DEEPX tutorial paths (identical first code cell in every notebook).
from pathlib import Path; import importlib, sys
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tutorial_paths.py").is_file())))
import tutorial_paths; importlib.reload(tutorial_paths)  # pick up updates without restarting the kernel
from tutorial_paths import setup_tutorial
import shlex

paths = setup_tutorial(needs=["dx_rt", "dx_app", "npu"])
paths.require(*["dx_rt", "dx_app", "npu"])   # stop here with the fix if a prerequisite is missing
%cd {DX_ALL_SUITE_DIR}


In [ ]:
!tree -L 1 dx-runtime

In [ ]:
!tree -L 1 dx-runtime/dx_app
!tree -L 1 dx-runtime/dx_app/bin
#!tree -L 2 dx-runtime/dx_app/src

**DX-APP** のデモは、最小限のセットアップで DEEPX NPU 上のコンパイル済みモデルを紹介できるよう最適化されています。
各デモは一般的な AI タスクを表しており、画像、動画、またはライブカメラ入力を使って
実行できます。

<img src="assets/ai-model-type.png" style="max-width: 1000px;">

---
**DX-APP リポジトリの構成:**
```shell
dx_app/
├── src/
│   ├── cpp_example/                    # C++ end-to-end examples (497 model variants across 29 tasks)
│   │   ├── <task>/<family>/<variant>/  #   One folder per .dxnn file: entry points, factory, config.json, README.md
│   │   ├── multi_model_graph/          #   Multi-model graph CLI and sample graph JSON files (section 4)
│   │   └── common/                     # Shared C++ runtime layer
│   │       ├── base/                   #   Abstract interfaces (factories, pre/post-processors, visualizers, inputs)
│   │       ├── processors/             #   Shared processors
│   │       ├── runner/                 #   Task-specific sync/async runner pairs
│   │       ├── inputs/                 #   Image, video, camera, and RTSP input sources
│   │       ├── visualizers/            #   Task-specific visualizers
│   │       ├── config/                 #   ModelConfig loader
│   │       ├── graph/                  #   Multi-model graph engine
│   │       ├── registry/               #   Model registry used by the graph engine
│   │       ├── trackers/               #   IoU tracker
│   │       ├── third_party/            #   Header-only third-party libraries
│   │       └── utility/                #   Labels, preprocessing, profiling, and run utilities
│   ├── python_example/                 # Python end-to-end examples (the same 497 variants and 29 tasks)
│   │   ├── <task>/<family>/<variant>/  #   Four scripts per variant, config.json, and the model card files
│   │   └── common/                     # Shared Python runtime layer (same structure as the C++ one)
│   ├── postprocess/                    # C++ post-processing (consumed by pybind11 bindings)
│   ├── utility/                        # Shared support code used by build flow
│   └── bindings/
│       └── python/
│           ├── dx_postprocess/         # pybind11 bindings wrapping src/postprocess/
│           └── dx_graph/               # pybind11 module running multi-model graphs (./build.sh --all)
├── config/
│   ├── model_registry.json             # Model registry: one row per variant, the single source of truth
│   ├── test_models.conf                # Test model configuration
│   └── README.md                       # Config directory documentation
├── scripts/                            # dx_tool.sh, extract_model_package.sh, download_models.py, and helpers
├── tests/                              # pytest-based test suites
├── assets/                             # Links to downloaded models and videos
├── sample/                             # Bundled images and task-specific sample data
├── bin/                                # C++ executables generated by the build: <variant>_sync, <variant>_async
├── build.sh                            # Build script (default: the 26 demo models; --all: every variant)
├── setup.sh                            # Model, video, and dependency setup
├── run_demo.sh                         # Demo launcher: 26 demos, up to 6 execution modes
├── run_multi_model_demo.sh             # Multi-model graph scenarios and graph checks
├── install.sh                          # Dependency and OpenCV installer
└── docs/                               # Detailed documentation
```
---
**タスク、ファミリー、バリアント**

すべてのサンプルは `<task>/<family>/<variant>/` に置かれています。**バリアント**は `.dxnn` ファイル名の拡張子を除いた部分であり、そのモデルに属するすべてのものの名前になります。

| 項目 | `yolov7_640x640.dxnn` の場合 |
|---|---|
| サンプルフォルダー | `src/cpp_example/object_detection/yolov7/yolov7_640x640/` と、`src/python_example/` 配下の同じパス |
| C++ 実行ファイル | `bin/yolov7_640x640_sync`、`bin/yolov7_640x640_async` |
| Python スクリプト | `yolov7_640x640_sync.py`、`yolov7_640x640_async.py`、`yolov7_640x640_sync_cpp_postprocess.py`、`yolov7_640x640_async_cpp_postprocess.py` |
| ダウンロード名 | `./setup.sh --models yolov7_640x640` |

> **DX-APP v3.2.x から移行する場合:** 実行ファイルはバリアント名で命名されるようになりました(`bin/yolov7_async` は `bin/yolov7_640x640_async` になりました)。PPU モデルは `object_detection/yolo_ppu/` に移動し、いくつかのタスクは名前が変更または分割されました。`classification` は `image_classification` に、`obb_detection` は `oriented_object_detection` に、`image_enhancement` は `low_light_enhancement` に、`face_alignment` は `face_landmark` になり、`embedding` は `face_recognition`、`image_retrieval`、`visual_place_recognition`、`zero_shot_image_classification` に分かれました。

**29 個のタスクディレクトリ**(`cpp_example` と `python_example` で共通):

- 検出: `object_detection`、`oriented_object_detection`、`face_detection`、`hand_detection`、`mainbody_detection`、`3d_object_detection`、`anomaly_detection`
- セグメンテーション: `instance_segmentation`、`semantic_segmentation`、`zero_shot_instance_segmentation`、`panoptic_driving_perception`、`image_matting`
- 姿勢とキーポイント: `pose_estimation`、`hand_landmark`、`face_landmark`、`keypoint_detection`、`object_pose_estimation`
- 分類と属性: `image_classification`、`zero_shot_image_classification`、`person_attribute`、`face_attribute`
- 認識と検索: `face_recognition`、`person_reid`、`image_retrieval`、`visual_place_recognition`
- 画像復元と深度: `image_denoising`、`super_resolution`、`low_light_enhancement`、`depth_estimation`

DX-APP v3.3.0 では、Model Zoo のモデルが新たに 147 個登録されました。RT-DETR(v1、v2、v3)、YOLOv12、YOLOv13、事前最適化済みの YOLO26、そして新しいタスク向けのモデルである異常検知(PatchCore、EfficientAD)、画像マッティング(PP-Matting)、CLIP による画像検索、人物再識別、視覚的場所認識(EigenPlaces)などが含まれます。

## 前提条件

- チュートリアル 01 を完了していること: DX-Runtime が `--all` でインストール済み(DX-RT、DX-APP、およびその `venv-dx-runtime` Python 環境)で、NPU が `/dev/dxrt0` として認識されている。
- **DX-RT 3.5.0 以降。** DX-APP v3.3.0 は Model Zoo リリース `2_5_0` からモデルをダウンロードします。その `.dxnn` ファイルはコンテナ形式 v9 を使用しており、それより古いランタイムでは読み込めません(トラブルシューティングを参照)。インストール済みのバージョンは `dxrt-cli --version` で確認できます。
- DX-APP のサンプルモデルと動画。チュートリアル 01 の 3.3 節でダウンロードします。以下のダウンロードセルでは、まだ不足しているものだけを取得します(このチュートリアル用のモデル約 300 MB と、新規ホストでは共有サンプル動画アーカイブ 1.2 GB)。
- グラフィカルなデスクトップセッションは任意です。セルはサンプルをヘッドレス(`--no-display --save`)で実行し、保存された画像をノートブック内に表示します。ウィンドウを開くセルは、`SHOW_WINDOW = False` のようなフラグで無効になっています。
- `sudo` は不要です。所要時間は約 45 分です。2 つのセルはそれぞれ 1〜2 分かかります。3.3 節の C++ ビルドと、グラフ実行ファイルがない場合の 4.1 節のビルドです。

次のセルは SDK の場所を特定し、これらの要件を確認してステータス表を表示します。`MISSING` と表示された項目には、それを提供する手順が併記されます。

## 1. モデルと動画の準備

1. `dx_app` ディレクトリに移動します。

In [ ]:
# Move to the configured DX-APP directory.
%cd {DX_APP_DIR}

2. `setup.sh` で必要なモデルとサンプル動画をダウンロードします。このスクリプトは対話型(どのモデルカテゴリをダウンロードするか尋ねます)なので、セルではなくターミナルで実行します。**File > New > Terminal** を開いて次を実行してください。

```bash
cd <DX_ALL_SUITE_DIR>/dx-runtime/dx_app
./setup.sh --no-force
```

3. 次のセルは、同じ 2 つのコマンドを実際のインストールパスで表示します。

In [ ]:
# Interactive script: run these two commands in a terminal (File > New > Terminal).
print(f"cd {shlex.quote(str(DX_APP_DIR))}")
print("./setup.sh --no-force")


> ![](assets/sc-setup.png)

4. Model Zoo 全体を保存できる空きストレージ(数十 GB)がある場合は、**Enter** を押してすべての Model Zoo モデルをダウンロードします。スクリーンショットは以前のリリースのもので、現在のメニューにはより多くのモデルと新しいタスク名が表示されます。

5. そうでない場合は、下のコードセルでこのチュートリアルに必要なモデルだけをダウンロードします。`yolov7_640x640` のような `.dxnn` ファイル名から拡張子を除いた**バリアント名**を指定してください。`YoloV7` のような以前の Model Zoo の名前も引き続き使用できます。チュートリアル 01 の 3.3 節では、これらのモデルをすでにダウンロードしています。完了済みであれば、`--no-force` によってセルはすべてのファイルをスキップし、数秒で終了します。第 3 節と第 4 節では、追加のモデルを同じ方法で取得します。

In [ ]:
# The models used in section 2, by variant name (the .dxnn file stem).
# --no-force skips files that already exist, so rerunning this cell is cheap.
SELECTED_MODELS = shlex.join([
    "resnet50_224x224",
    "yolov7_640x640",
    "yolov7_640x640_ppu",
    "yolov8-s-pose_640x640",
    "yolov8-n-seg_640x640",
])

# Preview what would be downloaded, without downloading:
#!cd "{DX_APP_DIR}" && bash setup.sh --models {SELECTED_MODELS} --dry-run

# Download the selected models only when they are missing
!cd "{DX_APP_DIR}" && bash setup.sh --models {SELECTED_MODELS} --no-force

6. モデルと動画の両方が期待どおりにダウンロードされたことを確認します。

In [ ]:
# Downloaded models and videos are stored under `assets` path as a symbolic link
!cd "{DX_APP_DIR}" && tree assets

In [ ]:
# AI models converted to DXNN format

# Number of downloaded models
!cd "{DX_APP_DIR}" && ls assets/models | wc -l

# Downloaded model list
!cd "{DX_APP_DIR}" && tree assets/models

In [ ]:
# video files for demo inputs
!cd "{DX_APP_DIR}" && tree assets/videos

## 2. デモの実行

**DX-APP** は、分類、検出、セグメンテーション、姿勢推定など、コンパイル済みモデルを DEEPX NPU 上で実行する方法を示す、すぐに使えるデモアプリケーション群です。

ユーザーは多様な DX-APP のデモを調べることで、アプリケーションの学習と構築ができます。

`bin/` 内の C++ 実行ファイルはバリアント名で命名されています(`bin/<variant>_sync` と `bin/<variant>_async`)。どれも同じ入力オプション(`-i` 画像、`-v` 動画、`-c` カメラ、`-r` RTSP ストリーム)と同じ出力オプションを受け付けます。

| オプション | 効果 |
|---|---|
| `--no-display` | ウィンドウを開かず、性能サマリーだけを表示する |
| `-s`、`--save` | 描画された結果を保存する |
| `--save-dir <dir>` | 保存先。実行するたびに新しいサブディレクトリが作成される |
| `-l`、`--loop <n>` | 入力を `n` 回繰り返す |

この節のセルは `--no-display --save` を付けてヘッドレスで実行し、保存された画像をセルの下に表示します。そのため SSH 経由でも動作し、ノートブックがブロックされることはありません。コメントアウトされた行は同じサンプルを動画またはカメラで実行します。これらはウィンドウを開くため、デスクトップセッションで使用してください。

次のセルは出力ディレクトリを作成し、最新の保存画像を表示する小さなヘルパーを用意します。

In [ ]:
import time
from IPython.display import Image, Markdown, display

DEMO_OUTPUT_DIR = WORKSPACE_DIR / "dx-app-outputs"   # results saved with --save in section 2
DEMO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Remember when each cell starts, so the helper shows only images saved by the current cell.
CELL_START = {"time": time.time()}
def _mark_cell_start(info):
    CELL_START["time"] = time.time()
ip = get_ipython()
for callback in list(ip.events.callbacks["pre_run_cell"]):
    if getattr(callback, "__name__", "") == "_mark_cell_start":
        ip.events.unregister("pre_run_cell", callback)
ip.events.register("pre_run_cell", _mark_cell_start)


def show_latest_image(folder, width=640):
    # Show the newest .jpg/.png that the current cell saved under `folder`
    # (every run saves into its own sub-directory).
    images = [p for p in Path(folder).rglob("*")
              if p.suffix.lower() in (".jpg", ".jpeg", ".png") and p.stat().st_mtime >= CELL_START["time"]]
    if not images:
        print(f"This cell saved no image under {folder}; check the command output above.")
        return
    latest = max(images, key=lambda p: p.stat().st_mtime)
    print("Saved result:", latest.relative_to(folder))
    display(Image(filename=str(latest), width=width))

### 2.1 分類
- **bin/resnet50_224x224_sync** は、ImageNet で学習済みの分類モデルを実行し、最も確率の高いクラスを画像上に描画します。

`-m` はすべての C++ サンプルで省略可能です。指定しない場合、サンプルは既定のモデル `assets/models/<variant>.dxnn` を読み込みます。モデルがどこから来るのかがわかるように、セルではあえて指定しています。

In [ ]:
%cd {DX_APP_DIR}

# Help: the options shared by every C++ example
!./bin/resnet50_224x224_sync -h

# Image input, headless: classify one ImageNet sample and save the annotated image
!./bin/resnet50_224x224_sync -m assets/models/resnet50_224x224.dxnn -i sample/ILSVRC2012/1.jpeg --no-display --save --save-dir "{DEMO_OUTPUT_DIR}"
show_latest_image(DEMO_OUTPUT_DIR)

### 2.2 物体検出
- **bin/yolov7_640x640_sync** と **bin/yolov7_640x640_async** は、YOLOv7 物体検出モデルを画像、動画、カメラに対して実行します。次のセルは、既定のビルドで作成されたその他の YOLO 実行ファイルを一覧表示します。

In [ ]:
%cd {DX_APP_DIR}

# Find other YOLO-based reference executables
!ls ./bin/yolo*

#### 実行方式

DX-APP は、アプリケーションの要件に合わせてさまざまな実行方式を提供します。

| 方式 | 主な特徴 | 推奨用途 |
|---|---|---|
| 同期 | 各フレームを順番に処理する | シンプルなフローとレイテンシの確認 |
| 非同期 | パイプラインの各ステージを重ねて実行する | より高いスループット |
| PPU | 対応する後処理を NPU 側にオフロードする | ホスト CPU の負荷軽減 |

- **同期アプリケーション**

<img src="https://github.com/DEEPX-AI/dx_rt/raw/main/docs/source/resources/09_01_Sync_Inference_Operation.png" style="max-width: 300px;">

In [ ]:
%cd {DX_APP_DIR}

# Image input, headless: the annotated image is saved and shown below
!./bin/yolov7_640x640_sync -m assets/models/yolov7_640x640.dxnn -i sample/img/sample_dog.jpg --no-display --save --save-dir "{DEMO_OUTPUT_DIR}"
show_latest_image(DEMO_OUTPUT_DIR)

# Video input, headless: 588 frames, prints the performance summary (about 20 seconds)
!./bin/yolov7_640x640_sync -m assets/models/yolov7_640x640.dxnn -v assets/videos/boat.mp4 --no-display

# On a desktop session (opens a window):
#!./bin/yolov7_640x640_sync -m assets/models/yolov7_640x640.dxnn -v assets/videos/boat.mp4
#!./bin/yolov7_640x640_sync -m assets/models/yolov7_640x640.dxnn -c 0

- **非同期アプリケーション**


<img src="https://github.com/DEEPX-AI/dx_rt/raw/main/docs/source/resources/09_01_Async_Inference_Operation.png" style="max-width: 1000px;">

次のセルは、同じ動画を非同期の実行ファイルで実行します。その `Overall FPS` の行を、上の同期実行の結果と比較してください。

In [ ]:
%cd {DX_APP_DIR}

# Video input, headless: compare "Overall FPS" with the synchronous run above
!./bin/yolov7_640x640_async -m assets/models/yolov7_640x640.dxnn -v assets/videos/boat.mp4 --no-display

# On a desktop session (opens a window):
#!./bin/yolov7_640x640_async -m assets/models/yolov7_640x640.dxnn -v assets/videos/boat.mp4
#!./bin/yolov7_640x640_async -m assets/models/yolov7_640x640.dxnn -c 0

# RTSP input (this public stream is reachable only from Korea)
#!./bin/yolov7_640x640_async -m assets/models/yolov7_640x640.dxnn -r "rtsp://210.99.70.120:1935/live/cctv002.stream"

- **PPU と非 PPU の比較**

以下の 2 つのセルは、同じ 7 秒のクリップを非 PPU モデル(`bin/yolov7_640x640_async`)と PPU モデル(`bin/yolov7_640x640_ppu_async`)で実行します。`SHOW_WINDOW = False`(既定)の場合は `--no-display` を付けてヘッドレスで実行し、FPS のサマリーだけを表示するため、ノートブックがブロックされることはありません。デスクトップセッションでは `SHOW_WINDOW = True` に設定すると、DX-APP のウィンドウで検出結果と FPS カウンターを確認できます。2 つの FPS の行を比較してください。PPU モデルは候補のフィルタリングとクラス選択を NPU 側に移しています。PPU のサンプルはすべて `object_detection/yolo_ppu/` にまとめられています。

In [ ]:
SHOW_WINDOW = False   # True opens the DX-APP window (desktop session only); False prints the FPS summary and returns
%cd {DX_APP_DIR}

# PPU disabled
!./bin/yolov7_640x640_async -m assets/models/yolov7_640x640.dxnn -v assets/videos/drone-city-road2.mov {"" if SHOW_WINDOW else "--no-display"}

In [ ]:
SHOW_WINDOW = False   # keep the same value as in the previous cell
%cd {DX_APP_DIR}

# PPU enabled
!./bin/yolov7_640x640_ppu_async -m assets/models/yolov7_640x640_ppu.dxnn -v assets/videos/drone-city-road2.mov {"" if SHOW_WINDOW else "--no-display"}

### 2.3 姿勢推定
- **bin/yolov8-s-pose_640x640_sync** は、姿勢推定モデルを実行し、画像、動画、カメラ内の人物を検出して体のキーポイントを推定します。

In [ ]:
%cd {DX_APP_DIR}

# Image input, headless
!./bin/yolov8-s-pose_640x640_sync -m assets/models/yolov8-s-pose_640x640.dxnn -i sample/img/sample_person_b.jpg --no-display --save --save-dir "{DEMO_OUTPUT_DIR}"
show_latest_image(DEMO_OUTPUT_DIR)

# On a desktop session (opens a window):
#!./bin/yolov8-s-pose_640x640_sync -m assets/models/yolov8-s-pose_640x640.dxnn -v assets/videos/dance-solo.mov
#!./bin/yolov8-s-pose_640x640_sync -m assets/models/yolov8-s-pose_640x640.dxnn -c 0

### 2.4 セグメンテーション
- **bin/yolov8-n-seg_640x640_async** は、インスタンスセグメンテーションモデルを実行し、画像、動画、カメラ内で見つかった各物体に対してボックスとピクセル単位のマスクを生成します。

In [ ]:
%cd {DX_APP_DIR}

# Image input, headless
!./bin/yolov8-n-seg_640x640_async -m assets/models/yolov8-n-seg_640x640.dxnn -i sample/img/sample_person_b.jpg --no-display --save --save-dir "{DEMO_OUTPUT_DIR}"
show_latest_image(DEMO_OUTPUT_DIR)

# On a desktop session (opens a window):
#!./bin/yolov8-n-seg_640x640_async -m assets/models/yolov8-n-seg_640x640.dxnn -v assets/videos/blackbox-city-road.mp4
#!./bin/yolov8-n-seg_640x640_async -m assets/models/yolov8-n-seg_640x640.dxnn -c 0

### 2.5 デモランチャー: run_demo.sh

`run_demo.sh` は、タスクごとに 1 つずつ、**26 個のデモ**を最大 **6 つの実行モード**(C++ の同期と非同期、および 4 つの Python バリアント)で起動します。デモ、モード、入力の 3 つを質問してくるので、ターミナル(**File > New > Terminal**)で起動してメニューに答えてください。

```bash
cd <DX_ALL_SUITE_DIR>/dx-runtime/dx_app
./run_demo.sh
```

次のセルは、同じコマンドを実際のパスで表示します。最初のメニューが表示される前に、ランチャーは 26 個のデモモデルのうち不足しているものをダウンロードするため、初回の起動には数分かかることがあります。また、`venv-dx-runtime` の Python を自動で見つけるため、Python モードでも仮想環境を有効化する必要はありません。

In [ ]:
# Interactive script: run these two commands in a terminal (File > New > Terminal).
print(f"cd {shlex.quote(str(DX_APP_DIR))}")
print("./run_demo.sh")


最初のメニューにはデモが一覧表示されます。

| # | デモ | # | デモ |
|---|---|---|---|
| 0 | Object Detection (YOLOv7) | 13 | Image Enhancement (Zero-DCE) |
| 1 | Object Detection (YOLOv11N) | 14 | Embedding (ArcFace) |
| 2 | Face Detection (SCRFD500M) | 15 | Attribute Recognition (DeepMAR-ResNet50) |
| 3 | OBB Detection (YOLO26N-OBB) | 16 | PPU Pipeline (YOLOv7-PPU) |
| 4 | Pose Estimation (YOLOv8s-Pose) | 17 | Keypoint Detection (SuperPoint) |
| 5 | Hand Landmark (HandLandmarkLite) | 18 | Object Pose Estimation (DOPE) |
| 6 | Face Alignment (3DDFA-V2-MobileNetV1) | 19 | Panoptic Driving (YOLOPv2) |
| 7 | Instance Segmentation (YOLOv8N-Seg) | 20 | 3D Object Detection (SFA3D) |
| 8 | Semantic Segmentation (DeepLabV3+MobileNet) | 21 | Hand Detection (MediaPipe Palm) |
| 9 | Classification (ResNet50) | 22 | Image Retrieval (CLIP RN50) |
| 10 | Depth Estimation (YOLO26-Depth-S) | 23 | Visual Place Recognition (EigenPlaces R18) |
| 11 | Image Denoising (DnCNN-50) | 24 | Person Re-ID (RepVGG-A0) |
| 12 | Super Resolution (ESPCN-X4) | 25 | Image Matting (PP-Matting HRNet-W48) |

2 つ目のメニューでは実行モードを選びます。モード 1 と 2 は C++ 実行ファイルを、モード 3〜6 は 3.4 節で説明する 4 つの Python バリアントを実行します。

> ![](assets/sc-run_demo2.png)

3 つ目のメニューでは入力を選びます。`1` はデモのサンプル動画、`2` はサンプル画像です。

3 つの回答はオプション `--task`、`--mode`、`--input` でも指定でき、その場合はすべてのメニューがスキップされます。次のセルはランチャーのヘルプを表示します。`RUN_DEMO_LAUNCHER = True` の場合は、続けてデモ `0`(YOLOv7)をモード `2`(C++ 非同期)でサンプル画像に対して起動します。デモはウィンドウを開くため、このフラグはデスクトップセッションでのみ設定してください。

In [ ]:
RUN_DEMO_LAUNCHER = False   # True starts the demo below; it opens a window (desktop session only)
%cd {DX_APP_DIR}

!./run_demo.sh --help

if RUN_DEMO_LAUNCHER:
    # Demo 0 (Object Detection, YOLOv7), mode 2 (cpp_async), input 2 (image): no menus
    !./run_demo.sh --task 0 --mode 2 --input 2

### 2.6 dx_tool.sh によるサンプルの検索、確認、エクスポート

`scripts/dx_tool.sh` はサンプルを管理するツールです。引数なしで起動すると対話型メニューが開きますが、検索系のコマンドは直接実行することもできるため、ノートブックで便利に使えます。

| コマンド | 内容 |
|---|---|
| `dx_tool.sh list [FILTER]` | サンプルを一覧表示する。タスクで絞り込むこともできる(例: `det`) |
| `dx_tool.sh search <KEYWORD>` | モデル名またはタスクでサンプルを検索する |
| `dx_tool.sh info <MODEL>` | 1 つのサンプルのファイル、ファクトリークラス、`config.json` を表示する |
| `dx_tool.sh run` / `dx_tool.sh bench` | 多数のサンプルを一度に実行またはベンチマークする(対話型フィルター) |
| `dx_tool.sh extract` | スタンドアロンパッケージをエクスポートする(対話型) |

次のセルは YOLO26-S のサンプルを検索し、検出モデルの詳細を表示します。`search` は各ヒットを `<task>/<family>/<variant>` の形式で表示します。これは第 3 節のエクスポートスクリプトが受け取る形式とまったく同じです。`info` は、`config.json` に記載された既定のサンプル画像と動画、スコアと NMS のしきい値などを表示します。

In [ ]:
%cd {DX_APP_DIR}

# Which examples exist for YOLO26-S? (C++ and Python, every task)
!./scripts/dx_tool.sh search yolo26-s

# Details of one variant: files, factory classes, and config.json
!./scripts/dx_tool.sh info yolo26-s_640x640

対話型メニューではエクスポートも行えます。質問に答える形式なので、ターミナル(File > New > Terminal)で実行してください。

```bash
cd <DX_ALL_SUITE_DIR>/dx-runtime/dx_app
./scripts/dx_tool.sh
```

次のセルは、同じコマンドを実際のパスで表示します。メニューが表示されたら、その下の手順に従ってください。

In [ ]:
# Interactive script: run these two commands in a terminal (File > New > Terminal).
print(f"cd {shlex.quote(str(DX_APP_DIR))}")
print("./scripts/dx_tool.sh")


メニューが表示されたら:
- `2`(Extract Package)を選択します。
- プログラミング言語を選択します: `1` C++、`2` Python、`3` 両方。
- サンプルのパスを `<task>/<family>/<variant>` の形式で入力します(例: `object_detection/yolo26/yolo26-s_640x640`)。Tab キーでパスを補完できます。
- Enter キーを押して、既定の出力ディレクトリ `output` を受け入れます。

> ![](assets/sc-extract.png)

スクリーンショットは、パスが 2 階層(`object_detection/yolo26s`)だった以前のリリースのものです。数秒後、パッケージが `output/cpp/<task>/<family>/<variant>` と `output/py/<task>/<family>/<variant>` に書き出されます。

このメニューは内部で `scripts/extract_model_package.sh` を呼び出しています。第 3 節ではこのスクリプトを直接使用します。モデルと言語を引数として受け取るため、ノートブックのセル内で実行できるからです。

## 3. 抽出したスタンドアロンサンプルの実行

2.6 節では、対話型のエクスポートを紹介しました。この節では、同じことをノートブックから `scripts/extract_model_package.sh` で行い、エクスポートしたパッケージを **DX-APP ツリーの外**で、自分のプロジェクトで行うのとまったく同じようにビルドして実行します。

| 節 | サンプル(`<task>/<family>/<variant>`) | 言語とバリアント | 示す内容 |
|---|---|---|---|
| 3.2 | `object_detection/yolo26/yolo26-s_640x640` | Python、`*_sync.py` | 任意の Model Zoo モデルを実行する最も簡単な方法 |
| 3.3 | `object_detection/yolo26/yolo26-s_640x640` | C++、`*_sync` | エクスポートした C++ パッケージを CMake でビルドする |
| 3.4 | `pose_estimation/yolo26_pose/yolo26-s-pose_640x640` | Python、`*_async_cpp_postprocess.py` | 別のタスクと、C++ による後処理 |
| 3.5 | `object_detection/yolo26/yolo26-s_640x640` | Python、4 つのバリアントすべて | どのバリアントを使うべきかの計測 |

エクスポートされた Python サンプルはすべて `dx_engine` と `dx_postprocess` をインポートします。これらは DX-Runtime が `dx-runtime/venv-dx-runtime` にインストールしたものです。そのため、セルはそのインタープリターを明示的に呼び出し、Python パッケージを `--no-prune` 付きでエクスポートします(理由は 3.1 節で説明します)。DX-APP ツリー内では、入力を指定しないとサンプルは既定のサンプル画像を使用しますが、エクスポートされたパッケージにはサンプル画像が含まれていません。そのため、セルは `dx_app/sample/img` の画像を渡してヘッドレス(`--no-display --save`)で実行し、保存された結果をノートブック内に表示します。

### 3.1 パッケージのエクスポート

まずスクリプト自身のヘルプから始めます。次のセルで使用する引数が一覧されます。サンプルは `<task>/<family>/<variant>`(ファミリーのすべてのバリアントを対象にする場合は `<task>/<family>`)、`--lang cpp|py|both`、`--output-dir`、そして `--no-prune` です。

In [ ]:
!cd "{DX_APP_DIR}" && bash scripts/extract_model_package.sh --help


`--output-dir` を指定しない場合、スクリプトは `dx_app` 内のその場でパッケージを準備します。指定した場合、パッケージは `<dir>/cpp/<task>/<family>/<variant>` と `<dir>/py/<task>/<family>/<variant>` にコピーされます。次のセルは 2 つの YOLO26 モデルをダウンロードし、3 つのパッケージをこのチュートリアルの `workspace/` ディレクトリ(git では無視されます)にエクスポートします。C++ の物体検出、そして Python の物体検出と姿勢推定です。

In [ ]:
STANDALONE_DIR = WORKSPACE_DIR / "standalone"
OUTPUT_DIR = WORKSPACE_DIR / "standalone-outputs"
DX_RUNTIME_PYTHON = DX_RUNTIME_DIR / "venv-dx-runtime" / "bin" / "python"   # has dx_engine and dx_postprocess
SAMPLE_IMG_DIR = DX_APP_DIR / "sample" / "img"
MODEL_DIR = DX_WORKSPACE_DIR / "res" / "models"

for path in (STANDALONE_DIR, OUTPUT_DIR):
    path.mkdir(parents=True, exist_ok=True)
if not DX_RUNTIME_PYTHON.is_file():
    raise FileNotFoundError(f"DX-Runtime Python environment not found: {DX_RUNTIME_PYTHON} (Tutorial 01, section 3)")

# Both models were downloaded in Tutorial 01 section 3.3; this only downloads what is missing.
!cd "{DX_APP_DIR}" && bash setup.sh --models yolo26-s_640x640 yolo26-s-pose_640x640 --no-force

# Export: object detection in C++ (pruned, the default) and in Python, pose estimation in Python.
# The Python packages use --no-prune; the next markdown cell explains why.
!cd "{DX_APP_DIR}" && bash scripts/extract_model_package.sh object_detection/yolo26/yolo26-s_640x640 --lang cpp --output-dir "{STANDALONE_DIR}"
!cd "{DX_APP_DIR}" && bash scripts/extract_model_package.sh object_detection/yolo26/yolo26-s_640x640 --lang py --output-dir "{STANDALONE_DIR}" --no-prune
!cd "{DX_APP_DIR}" && bash scripts/extract_model_package.sh pose_estimation/yolo26_pose/yolo26-s-pose_640x640 --lang py --output-dir "{STANDALONE_DIR}" --no-prune

!tree -L 5 "{STANDALONE_DIR}"

#### プルーニングされたパッケージと `--no-prune`

既定では、エクスポートは `common/` を**プルーニング**します。つまり、ランタイム層全体ではなく、このサンプルが実際に使用する共有ファイルだけをコピーします。パッケージは小さく読みやすくなりますが、このサンプルしか実行できません。`--no-prune` は `common/` 全体をコピーします。同じパッケージに 2 つ目のモデルを追加する予定がある場合も、こちらが必要です。前のセルの C++ パッケージはプルーニングされています。次のセルは Python の YOLO26-S サンプルを既定のプルーニングでもう一度エクスポートし、完全なコピーと比較します。

> **DX-APP v3.3.0 に関する注:** プルーニングされた **Python** パッケージには、その `config.json` が指定するプロセッサクラスが欠けているため、起動時に `VariantConfigError: processor class 'LetterboxPreprocessor' is exported by neither common.processors nor common.visualizers` で停止します。この問題が修正されるまでは、このチュートリアルと同様に Python パッケージを `--no-prune` 付きでエクスポートしてください。プルーニングされた C++ パッケージには影響しません。

In [ ]:
PRUNED_DIR = WORKSPACE_DIR / "standalone-pruned"
!cd "{DX_APP_DIR}" && bash scripts/extract_model_package.sh object_detection/yolo26/yolo26-s_640x640 --lang py --output-dir "{PRUNED_DIR}"


def count_files(folder):
    return sum(1 for p in Path(folder).rglob("*") if p.is_file())


PACKAGE = Path("py") / "object_detection" / "yolo26" / "yolo26-s_640x640"
print(f"\n{'package':<14}{'all files':>10}{'common/ files':>15}")
for label, root in (("pruned", PRUNED_DIR), ("--no-prune", STANDALONE_DIR)):
    folder = root / PACKAGE
    print(f"{label:<14}{count_files(folder):>10}{count_files(folder / 'common'):>15}")

#### モデルカード

すべてのサンプルフォルダーには、Model Zoo から生成された**モデルカード**が含まれています。`README.md` はモデルについて説明しており、モデルの概要、出典付きの精度と速度、ダウンロード、実行方法が記載されています。Python のサンプルフォルダーには、ブラウザでオフライン閲覧できる同じカードである `model.html` と、参照用の結果 `result.webp` もあります。エクスポートされた Python パッケージはこの 3 つをすべて保持します。次のセルは、先ほどエクスポートしたパッケージの参照画像とカードを表示します。

In [ ]:
import re

CARD_DIR = STANDALONE_DIR / "py" / "object_detection" / "yolo26" / "yolo26-s_640x640"
!ls "{CARD_DIR}"

display(Image(filename=str(CARD_DIR / "result.webp"), width=480))   # the reference result
card = (CARD_DIR / "README.md").read_text()
display(Markdown(re.sub(r"!\[[^\]]*\]\(result\.webp\)", "", card)))  # the card; its image is shown above
print("Offline HTML card:", CARD_DIR / "model.html")

### 3.2 例 1: Python スタンドアロン、物体検出

`yolo26-s_640x640_sync.py` は、純粋な Python の後処理で 1 フレームずつ処理する、プレーンな同期バリアントです。最初に読むべきバリアントです。引数は DX-APP のすべての Python サンプルで共通です。`--model`、入力オプションのいずれか 1 つ(`--image`、`--video`、`--camera`、`--rtsp`)、そしてヘッドレス実行用の `--no-display --save` です。

```bash
cd <workspace>/standalone/py/object_detection/yolo26/yolo26-s_640x640
<dx-runtime>/venv-dx-runtime/bin/python yolo26-s_640x640_sync.py \
    --model <workspace>/res/models/yolo26-s_640x640.dxnn \
    --image <dx_app>/sample/img/sample_street.jpg \
    --no-display --save --save-dir <output-dir>
```

In [ ]:
EXAMPLE_DIR = STANDALONE_DIR / "py" / "object_detection" / "yolo26" / "yolo26-s_640x640"
MODEL = MODEL_DIR / "yolo26-s_640x640.dxnn"
IMAGE = SAMPLE_IMG_DIR / "sample_street.jpg"

!cd "{EXAMPLE_DIR}" && "{DX_RUNTIME_PYTHON}" yolo26-s_640x640_sync.py --model "{MODEL}" --image "{IMAGE}" --no-display --save --save-dir "{OUTPUT_DIR}"

show_latest_image(OUTPUT_DIR)

### 3.3 例 2: C++ スタンドアロン、物体検出

エクスポートされた C++ パッケージには、`yolo26-s_640x640_sync.cpp`、`yolo26-s_640x640_async.cpp`、共有の `common/` 層、そしてインストール済みの DX-RT にリンクする生成済みの `CMakeLists.txt` が含まれています。ビルドは標準的な CMake の手順で、DX-APP ツリーを参照するものは何もありません。

```bash
cd <workspace>/standalone/cpp/object_detection/yolo26/yolo26-s_640x640
mkdir -p build && cd build
cmake .. && make -j$(nproc)
./yolo26-s_640x640_sync -m <workspace>/res/models/yolo26-s_640x640.dxnn \
    -i <dx_app>/sample/img/sample_street.jpg --no-display --save --save-dir <output-dir>
```

C++ サンプルではオプションが `-m/--model_path` と `-i/--image_path` と表記される点に注意してください。`./yolo26-s_640x640_sync --help` で一覧できます。ビルドには 1〜2 分かかります。

In [ ]:
CPP_DIR = STANDALONE_DIR / "cpp" / "object_detection" / "yolo26" / "yolo26-s_640x640"

!cd "{CPP_DIR}" && mkdir -p build && cd build && cmake .. && make -j$(nproc)

!cd "{CPP_DIR}/build" && ./yolo26-s_640x640_sync -m "{MODEL}" -i "{IMAGE}" --no-display --save --save-dir "{OUTPUT_DIR}"

show_latest_image(OUTPUT_DIR)

### 3.4 例 3: Python スタンドアロン、姿勢推定、C++ 後処理による非同期実行

すべての Python サンプルには 4 つのバリアントがあります。ファイル名を見れば、どれを実行しているかがわかります。

| バリアント | 後処理 | スレッド方式 | 用途 |
|---|---|---|---|
| `*_sync.py` | 純粋な Python | 同期 | ロジックの学習とデバッグ |
| `*_async.py` | 純粋な Python | 非同期 | より高いスループット。デプロイの一般的な出発点 |
| `*_sync_cpp_postprocess.py` | C++ バインディング(`dx_postprocess`) | 同期 | 1 フレームずつ処理する重い後処理 |
| `*_async_cpp_postprocess.py` | C++ バインディング(`dx_postprocess`) | 非同期 | 高フレームレートでの重い後処理 |

> DX-APP v3.2.x では、`*_cpp_postprocess.py` スクリプトが実際には Python の後処理を実行していました。v3.3.0 でこれが修正されました。そのため、古いリリースで計測した結果ではバリアント間に差が出ません。

この例では、別のタスクと、非同期の C++ 後処理バリアントに切り替えます。コマンドの形は変わりません。変わるのはディレクトリ、スクリプト名、モデルだけです。

```bash
cd <workspace>/standalone/py/pose_estimation/yolo26_pose/yolo26-s-pose_640x640
<dx-runtime>/venv-dx-runtime/bin/python yolo26-s-pose_640x640_async_cpp_postprocess.py \
    --model <workspace>/res/models/yolo26-s-pose_640x640.dxnn \
    --image <dx_app>/sample/img/sample_people.jpg \
    --no-display --save --save-dir <output-dir>
```

In [ ]:
POSE_DIR = STANDALONE_DIR / "py" / "pose_estimation" / "yolo26_pose" / "yolo26-s-pose_640x640"
POSE_MODEL = MODEL_DIR / "yolo26-s-pose_640x640.dxnn"
POSE_IMAGE = SAMPLE_IMG_DIR / "sample_people.jpg"

!cd "{POSE_DIR}" && "{DX_RUNTIME_PYTHON}" yolo26-s-pose_640x640_async_cpp_postprocess.py --model "{POSE_MODEL}" --image "{POSE_IMAGE}" --no-display --save --save-dir "{OUTPUT_DIR}"

show_latest_image(OUTPUT_DIR)

### 3.5 バリアントを選ぶ前に計測する

4 つのバリアントのうちどれが最速かは、どこに時間がかかっているかによって決まります。環境変数 `DXAPP_PROGRESS=1` を設定すると、すべてのサンプルが約 1 秒ごとに `[PROGRESS]` 行を出力し、最後にも 1 行を出力します。各行には、スループット(`fps`、`avg_fps`)と、各ステージの 1 フレームあたりの平均時間が含まれます。入力の読み込み(`read_ms`)、前処理(`pre_ms`)、NPU 推論(`infer_ms`)、後処理(`post_ms`)です。

次のセルは、4 つの YOLO26-S バリアントすべてを同じ 588 フレームの動画でヘッドレス実行し、それぞれの最終行を表にまとめます。1 回の実行は次のようになります。

```bash
DXAPP_PROGRESS=1 <dx-runtime>/venv-dx-runtime/bin/python yolo26-s_640x640_sync.py \
    --model <workspace>/res/models/yolo26-s_640x640.dxnn --video <dx_app>/assets/videos/boat.mp4 --no-display
```

このセルには約 40 秒かかります。

In [ ]:
VIDEO = DX_APP_DIR / "assets" / "videos" / "boat.mp4"
VARIANT_SCRIPTS = [
    "yolo26-s_640x640_sync.py",
    "yolo26-s_640x640_async.py",
    "yolo26-s_640x640_sync_cpp_postprocess.py",
    "yolo26-s_640x640_async_cpp_postprocess.py",
]

rows = []
for script in VARIANT_SCRIPTS:
    log = !cd "{EXAMPLE_DIR}" && DXAPP_PROGRESS=1 "{DX_RUNTIME_PYTHON}" {script} --model "{MODEL}" --video "{VIDEO}" --no-display 2>&1
    progress = [line for line in log if line.startswith("[PROGRESS]")]
    if not progress:
        print("\n".join(log[-20:]))
        raise RuntimeError(f"{script} printed no [PROGRESS] line; see its output above")
    rows.append((script, dict(re.findall(r"(\w+)=([\d.]+)", progress[-1]))))

print(f"{'variant':<44}{'avg_fps':>9}{'pre_ms':>8}{'infer_ms':>10}{'post_ms':>9}")
for script, m in rows:
    print(f"{script:<44}{m['avg_fps']:>9}{m['pre_ms']:>8}{m['infer_ms']:>10}{m['post_ms']:>9}")

表の読み方(数値はホスト CPU と NPU カードによって異なります):

- **非同期と同期の比較:** 非同期バリアントは NPU 上で複数のフレームを同時に処理し続けるため、はるかに高いフレームレートに達します。DX-M1 カード 1 枚の場合、YOLO26-S は非同期で約 200 FPS、同期で約 50 FPS で動作します。非同期モードで `infer_ms` が大きくなるのは、NPU の処理時間だけでなく、各フレームを投入してから結果が得られるまでの往復時間を計測しているためです。
- **C++ と Python の後処理の比較:** `cpp_postprocess` バリアントは `post_ms` を短縮しますが、YOLO26-S では後処理が 1 ミリ秒を大きく下回るため、フレームレートはほとんど変わりません。C++ の後処理が効果を発揮するのは、セグメンテーションマスク、多数のアンカー、低速なホスト CPU など、`post_ms` がフレーム時間の大部分を占める場合です。
- **目安:** まず `*_async.py` から始めて `DXAPP_PROGRESS=1` で計測し、後処理がボトルネックになっている場合にのみ `cpp_postprocess` バリアントに切り替えてください。

### 3.6 試してみよう: Model Zoo の任意のモデル

これまでの例は 1 つのパターンに従っており、これは 497 個すべてのバリアントに当てはまります。

1. [Model Zoo](https://developer.deepx.ai/modelzoo/) のページでモデルを選び、`./scripts/dx_tool.sh search <keyword>` でそのサンプルを見つけます。
2. その場で実行します。DX-APP ツリー内では、モデルと入力は省略可能です。`--model` を指定しない場合、サンプルは `assets/models/<variant>.dxnn` を読み込み(不足している場合は先にダウンロードします)、入力を指定しない場合はそのタスクの既定のサンプル画像を使用します。
3. それをもとに開発したい場合はエクスポートします: `scripts/extract_model_package.sh <task>/<family>/<variant> --lang py --output-dir <dir>`

次のセルは `config/model_registry.json` でバリアントを検索し、その同期 Python サンプルをモデルも入力も指定せずにその場で実行します。これが最も短い実行方法です。セルは結果を表示し、エクスポートコマンドを出力します。`VARIANT` を変更して、例えば `resnet50_224x224`(分類)、`yolov12-n_640x640`(YOLOv12)、`rtdetr-r18vd-6x_640x640`(RT-DETR)など、別のモデルを試してください。まだディスク上にないモデルは先にダウンロードされ、数秒から 1 分ほどかかります。

In [ ]:
import json

VARIANT = "scrfd-500m_640x640"   # change me: any variant (the .dxnn file stem) from config/model_registry.json

registry = json.loads((DX_APP_DIR / "config" / "model_registry.json").read_text())
entry = next((e for e in registry if VARIANT in (e["variant"], e["model_name"])), None)
if entry is None:
    raise KeyError(f"{VARIANT} is not in the registry. Search with: cd {DX_APP_DIR} && ./scripts/dx_tool.sh search {VARIANT}")

task, family, variant = entry["task"], entry["family"], entry["variant"]
SCRIPT = f"src/python_example/{task}/{family}/{variant}/{variant}_sync.py"
print(f"{variant}: task={task}, family={family}, model file={entry['dxnn_file']}\n")

# The shortest run: no --model and no input, so the example uses its default model and sample image.
!cd "{DX_APP_DIR}" && "{DX_RUNTIME_PYTHON}" {SCRIPT} --no-display --save --save-dir "{OUTPUT_DIR}"
show_latest_image(OUTPUT_DIR)

print("\nExport it as a standalone package:")
print(f'cd "{DX_APP_DIR}" && bash scripts/extract_model_package.sh {task}/{family}/{variant} --lang py --output-dir "{STANDALONE_DIR}"')

## 4. マルチモデルグラフ: コードを書かずにモデルを連結する

実際のアプリケーションでは、1 フレームに対して複数のモデルを実行することがよくあります。例えば、検出モデルが人物を見つけて 2 つ目のモデルがその属性を読み取ったり、手のひら検出モデルが手を見つけてランドマークモデルが指の関節を配置したりします。DX-APP v3.3.0 では、このための**マルチモデルグラフ**が追加されました。パイプラインを JSON ファイルで記述すれば、1 つの実行ファイルがそれを実行します。

- **ノード**は入力ソースとモデルで、各モデルはバリアント名で指定します。
- **エッジ**はノード同士をつなぎます。`"roi"` を持つエッジは、最初のモデルが見つけた各ボックスを切り出し、その切り出し画像を次のモデルに渡します。

編集すべき C++ コードも、再ビルドも必要ありません。別のモデルを試すには、JSON ファイル内の文字列を 1 つ変更するだけです。実行ファイルは `bin/multi_model_graph_sync` と `bin/multi_model_graph_async` で、サンプルグラフは `src/cpp_example/multi_model_graph/` にあります。

| オプション | 効果 |
|---|---|
| `--graph <file>` | グラフを実行する |
| `--check <file>` | NPU を使わずに、グラフとそのモデルファイルを検証する |
| `--output <file>` | 合成された結果画像を保存する |
| `--report <file>` | すべてのノードの結果を JSON として保存する |
| `--list-models` | グラフで使用できるモデルを一覧表示する(`--consumes roi` で 2 段目のモデルを一覧表示) |

> Python モジュール `dx_graph` を使うと、同じグラフを Python から実行できます。このモジュールは `./build.sh --all` でのみビルドされるため、このチュートリアルでは実行ファイルを使用します。

### 4.1 グラフ実行ファイルのビルドとモデルのダウンロード

次のセルは、DX-APP のビルドにまだ 2 つの実行ファイルが含まれていない場合にそれらをビルドします。これには約 1 分かかります。続いて、この節のグラフで使用する 3 つの追加モデル(手のひら検出モデル、手のランドマークモデル、人物属性モデル)をダウンロードします。

In [ ]:
%cd {DX_APP_DIR}
GRAPH_DIR = DX_APP_DIR / "src" / "cpp_example" / "multi_model_graph"
GRAPH_OUTPUT_DIR = WORKSPACE_DIR / "graph-outputs"
GRAPH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not (DX_APP_DIR / "bin" / "multi_model_graph_async").is_file():
    !./build.sh --target multi_model_graph_sync multi_model_graph_async
!ls bin/multi_model_graph_*

!bash setup.sh --models mediapipe-hand-detector_192x192 mediapipe-hands-lite_224x224 deepmar_resnet50_224x224 --no-force

### 4.2 2 段のカスケード: 手のひら検出から手のランドマークへ

`hand_cascade.json` は最も小さなサンプルグラフです。

In [ ]:
!cat "{GRAPH_DIR}/hand_cascade.json"

ソース `cam` はサンプル画像です。`palm` はフレーム全体から手を見つけます。`palm` から `landmark` へのエッジには `"roi"` があるため、各手のボックスが 15% の余白を付けて切り出され、ランドマークモデルに渡されます。

`--check` は、NPU 上で何かを実行する前にグラフを検証します。各モデルを解決し、それが `produces` するものと `consumes` するものを表示し、`.dxnn` ファイルが存在することを確認します。続いて `--graph` でグラフを実行し、`--output` で合成画像を保存し、`--report` ですべてのノードの結果を書き出します。サンプルグラフは `dx_app` からの相対パスを使用しているため、`dx_app` から実行してください。

In [ ]:
%cd {DX_APP_DIR}

!./bin/multi_model_graph_async --check src/cpp_example/multi_model_graph/hand_cascade.json

!./bin/multi_model_graph_async --graph src/cpp_example/multi_model_graph/hand_cascade.json --output "{GRAPH_OUTPUT_DIR}/hand_cascade.jpg" --report "{GRAPH_OUTPUT_DIR}/hand_cascade.json"

display(Image(filename=str(GRAPH_OUTPUT_DIR / "hand_cascade.jpg"), width=480))

レポートには、フレームごとに各ノードの結果が記録されます。フレーム全体に対して実行されたモデルは `nodes` の下に、切り出し画像に対して実行されたモデルは `roi_nodes` の下に、各切り出し画像の元となったフレーム内のボックスとともに記録されます。次のセルは最初のフレームの内容をまとめて表示します。

In [ ]:
report = json.loads((GRAPH_OUTPUT_DIR / "hand_cascade.json").read_text())
frame = report["frames"][0]
for name, node in frame["nodes"].items():
    payload = node["payload"]
    print(f"{name:<10} full frame  {payload['shape']:<10} items={len(payload.get('items', []))}")
for name, crops in frame.get("roi_nodes", {}).items():
    for crop in crops:
        box = [round(v) for v in crop["origin"]["src_box"]]
        print(f"{name:<10} crop        {crop['payload']['shape']:<10} from box {box}")

### 4.3 JSON を編集してモデルを差し替える

`cascade_od_attr.json` は人物を検出し、切り出した各人物画像を人物属性モデル `deepmar_resnet50_224x224`(性別、年齢層、服装など)に渡します。このグラフの検出モデルは `yolov8-n_640x640` ですが、このチュートリアルではダウンロードしていません。次のセルは、それをダウンロードする代わりに、検出モデルを第 3 節のモデル `yolo26-s_640x640` に置き換えたグラフのコピーをワークスペースに書き出します。コピーは `dx_app` の外に置かれるため、セルはソース画像のパスも絶対パスにします。その後、編集したグラフを検証して実行します。再ビルドは一切行いません。

In [ ]:
graph = json.loads((GRAPH_DIR / "cascade_od_attr.json").read_text())
for node in graph["nodes"]:
    if node.get("type") == "source":
        node["uri"] = str(DX_APP_DIR / node["uri"])   # absolute path: the copy is outside dx_app
    if node["id"] == "od":
        node["model"] = "yolo26-s_640x640"           # the one-string model swap

MY_GRAPH = GRAPH_OUTPUT_DIR / "person_attributes_yolo26.json"
MY_GRAPH.write_text(json.dumps(graph, indent=2) + "\n")
!cat "{MY_GRAPH}"

!./bin/multi_model_graph_async --check "{MY_GRAPH}"
!./bin/multi_model_graph_async --graph "{MY_GRAPH}" --output "{GRAPH_OUTPUT_DIR}/person_attributes.jpg"

display(Image(filename=str(GRAPH_OUTPUT_DIR / "person_attributes.jpg"), width=640))

### 4.4 適合するモデルと、その他の機能

すべてのモデルがすべての位置に当てはまるわけではありません。2 段目のモデルは切り出し画像を受け付ける必要があり、その `consumes` 列が `roi` または `either` でなければなりません。`--list-models --consumes roi` はレジストリからこれらのモデルを抽出します。`file` 列は `.dxnn` がディスク上にあるかどうかを示します。次のセルは最初の数行と総数を表示します。

続いて `run_multi_model_demo.sh --list` でサンプルグラフを一覧表示します。サンプルグラフには、カスケード(検出モデルが属性、再識別、分類のモデルに結果を渡す)、ファンアウト(同じフレームに対する検出、セグメンテーション、姿勢推定)、チェーン(検出の前に超解像や低照度補正を行う)、そしてフィルタリング、カウント、追跡、ゾーンやラインの判定を行うロジックノードが含まれます。`bash run_multi_model_demo.sh --check` は NPU を使わずにすべてのサンプルグラフを検証し、不足しているモデルごとにダウンロードコマンドを表示します。

In [ ]:
%cd {DX_APP_DIR}

# Second-stage candidates: the first lines, then the number of models
!./bin/multi_model_graph_async --list-models --consumes roi | head -12
!./bin/multi_model_graph_async --list-models --consumes roi | tail -n +3 | wc -l

# The sample graphs and the demo scenarios
!bash run_multi_model_demo.sh --list

# Validate every sample graph (no NPU use; long output):
#!bash run_multi_model_demo.sh --check

## 5. トラブルシューティング

| 症状 | 表示される内容 | 考えられる原因 | 対処 |
|---|---|---|---|
| 以前の実行ファイル名が見つからない | `./bin/yolov7_async: No such file or directory` | DX-APP v3.3.0 では実行ファイルがバリアント名で命名される | `./bin/yolov7_640x640_async` のような新しい名前を使う。`ls bin` で一覧できる |
| 実行ファイルがない | `./bin/<variant>_sync: No such file or directory` | 既定のビルド(`--minimal`)は 26 個のデモモデルだけをビルドする | ターミナルで `dx_app` 内の `./build.sh --target <variant>` または `./build.sh --all` を実行する |
| 読み込み時にモデルが拒否される | `.dxnn container v9 needs DX-RT >= 3.5.0, but this runtime is 3.4.1` または `Model file format version 9 is not supported` | Model Zoo `2_5_0` のファイルには DX-RT 3.5.0 以降が必要 | DX-Runtime を更新する(チュートリアル 01 の第 3 節)。インストール済みのバージョンは `dxrt-cli --version` で確認できる |
| `import dx_engine` が失敗する | `dx_engine and DX-RT versions do not match` | `venv-dx-runtime` 内の `dx_engine` が別の DX-RT 向けにビルドされている | 両方が 3.5.x になるように DX-Runtime を再インストールする |
| モデルを読み込めない | `assets/models` 配下のパスを含む `Failed to load model` | モデルがダウンロードされていない | 第 1 節のダウンロードセルをバリアント名で実行し、再実行する |
| グラフの検証でモデルの不足が報告される | `MISSING: 1 model file is not in .../assets/models` | グラフがダウンロードされていないモデルを指定している | `--check` が表示するダウンロードコマンド、または `bash setup.sh --models <variant> --no-force` を実行する |
| 出力ウィンドウが表示されない | `cannot open display`、またはウィンドウなしでセルが終了する | グラフィカルセッションがない(SSH またはヘッドレスホスト) | デスクトップセッションで実行するか、`--no-display --save` のまま保存された結果を確認する |
| カメラを開けない | `Could not open camera 0` | `/dev/video0` がないか、ユーザーが `video` グループに属していない | `v4l2-ctl --list-devices` を確認し、`sudo usermod -aG video $USER` を実行して再ログインする |
| `run_demo.sh` がメニューの前に停止する | `Failed to download demo models.` | デモモデルの自動ダウンロード中にネットワークに接続できない | ネットワークに接続するか、ネットワークに接続できる状態で `./setup.sh --demo-models` を一度実行する |
| エクスポートした Python サンプルが起動時に停止する | `VariantConfigError: processor class '...' is exported by neither common.processors nor common.visualizers` | パッケージが既定のプルーニングでエクスポートされ、`config.json` が指定するクラスが欠けている(DX-APP v3.3.0) | `--no-prune` を付けて再度エクスポートする |
| エクスポートした Python サンプルで `dx_engine` をインポートできない | `ModuleNotFoundError: No module named 'dx_engine'` | DX-Runtime の環境ではなく Jupyter の Python で実行された | 第 3 節のセルと同様に `<dx-runtime>/venv-dx-runtime/bin/python` を使う |
| エクスポートした C++ パッケージの構成に失敗する | `Could not find a package configuration file provided by "dxrt"` | DX-RT がインストールされていないか、カスタムプレフィックス配下にある | DX-Runtime をインストールする(チュートリアル 01 の第 3 節)か、`cmake` の行に `-DCMAKE_PREFIX_PATH=<prefix>` を追加する |
| エクスポートしたサンプルが入力を見つけられない | `Input file not found: sample/img/...` | パッケージにサンプル画像が含まれていない | `dx_app/sample/img` のファイルを `--image`(Python)または `-i`(C++)で渡す |

## 6. まとめ

### 6.1 完了した DX-APP ワークフロー

```text
Load SDK paths
       │
       ▼
Download the selected models by variant name
       │
       ▼
Run C++ or Python examples headless and view the saved results
       │
       ├── Synchronous
       ├── Asynchronous
       └── PPU
       │
       ▼
Find an example with dx_tool.sh, export it, and measure its variants
       │
       ▼
Chain several models with a multi-model graph
```

### 6.2 学習ダッシュボード

| 領域 | できるようになったこと |
|---|---|
| SDK パス | `config.json` から `DX_APP_DIR` と関連パスを読み込む |
| 命名規則 | `<task>/<family>/<variant>` のバリアント名から、サンプル、その実行ファイル、スクリプトを見つける |
| モデルのセットアップ | 演習に必要なモデルと動画だけを、バリアント名でダウンロードする |
| 実行方式 | 目的に応じて同期、非同期、PPU の実行方式を選ぶ |
| 計測 | 1 つを選ぶ前に、`DXAPP_PROGRESS=1` で 4 つの Python バリアントを比較する |
| デモランチャー | `run_demo.sh` でデモを起動する。メニューから、または `--task`、`--mode`、`--input` で指定する |
| スタンドアロンコード | `scripts/extract_model_package.sh` で任意のモデルのサンプルをエクスポートし、そのモデルカードを読み、DX-APP の外でビルドして実行する |
| Model Zoo の網羅性 | `config/model_registry.json` または `dx_tool.sh search` で任意の Model Zoo モデルに対応する DX-APP サンプルを見つける |
| マルチモデルグラフ | 複数のモデルを連結するグラフ JSON を検証、実行、編集する |

### 6.3 完了チェックリスト

- [ ] DEEPX NPU が DX-RT から認識されており、DX-RT が 3.5.0 以降であることを確認した
- [ ] 必要な DX-APP のモデルと動画アセットを準備した
- [ ] エンドツーエンドの推論サンプルを少なくとも 1 つ実行し、保存された結果を確認した
- [ ] 利用可能な実行方式を比較した
- [ ] `dx_tool.sh search` で対象タスクのリファレンス実装を見つけた
- [ ] `run_demo.sh` を開いた
- [ ] スタンドアロンの Python と C++ サンプルを DX-APP ツリーの外でエクスポート、ビルド、実行した
- [ ] 4 つの Python バリアントを計測し、自分のモデルに合うものを選んだ
- [ ] 自分で選んだ Model Zoo モデルを実行した
- [ ] マルチモデルグラフを実行し、そのモデルの 1 つを差し替えた

> **次へ:** チュートリアル 03 に進み、学習済みモデルを調整して DXNN にコンパイルし、DX-APP のランタイム設定に統合してみましょう。